# Atlas Forge Engine 0.4 API — T4 GPU

Versão configurada para rodar no **Google Colab com GPU T4** como uma API temporária.

> O notebook solicita GPU ao abrir. Se o Colab não conceder uma T4 automaticamente, use **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU** e execute novamente a primeira célula.

Rotas expostas:
- `POST /image-to-3d`
- `GET /image-to-3d-status/{job_id}`
- `GET /models/{job_id}`


## 1. Confirmar GPU T4


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise RuntimeError(
        'Esta sessão abriu sem GPU. No Colab, selecione: '
        'Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU. '
        'Depois reconecte e rode novamente esta célula.'
    )

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, check=True
)
gpu_info = gpu.stdout.strip()
print('GPU disponível:', gpu_info)

if 'T4' not in gpu_info.upper():
    raise RuntimeError(
        f'GPU atual não é T4: {gpu_info}. Abra Alterar tipo de ambiente de execução e selecione T4 GPU.'
    )

print('✅ T4 confirmada. Pode seguir para a etapa 2.')


## 2. Preparar o motor isolado


In [ ]:
import pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/TripoSR')
ENV = pathlib.Path('/content/atlas_forge_env')
JOBS_ROOT = pathlib.Path('/content/atlas_jobs')
JOBS_ROOT.mkdir(parents=True, exist_ok=True)
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSR.git', str(ROOT)], check=True)
if ENV.exists():
    shutil.rmtree(ENV)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'virtualenv'], check=True)
subprocess.run([sys.executable, '-m', 'virtualenv', '--system-site-packages', str(ENV)], check=True)
PYTHON = str(ENV / 'bin' / 'python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '--ignore-installed', 'numpy==1.26.4', 'cupy-cuda12x==13.3.0', 'onnxruntime==1.20.1', 'rembg==2.0.61', 'omegaconf==2.3.0', 'Pillow==10.1.0', 'einops==0.7.0', 'transformers==4.35.0', 'trimesh==4.0.5', 'huggingface-hub==0.17.3', 'imageio[ffmpeg]', 'xatlas==0.0.9', 'moderngl==5.10.0', 'pygltflib', 'fastapi', 'uvicorn[standard]', 'python-multipart', 'pyngrok'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git@3aef8afa5f21b113afc4f4ea148baee850cbd472'], check=True)
print('Atlas Forge Engine 0.4 preparado.')


## 3. Criar a API do motor


In [ ]:
from pathlib import Path
APP_FILE = Path('/content/atlas_forge_api.py')
APP_FILE.write_text('''
import json, os, shutil, subprocess, threading, uuid
from pathlib import Path
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
ROOT = Path('/content/TripoSR')
PYTHON = str(Path('/content/atlas_forge_env/bin/python'))
JOBS_ROOT = Path('/content/atlas_jobs')
JOBS_ROOT.mkdir(parents=True, exist_ok=True)
app = FastAPI(title='Atlas Forge Engine API', version='0.4')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=False, allow_methods=['*'], allow_headers=['*'])
def write_status(job_dir, payload):
    (job_dir/'status.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
def read_status(job_id):
    p=JOBS_ROOT/job_id/'status.json'
    if not p.exists(): raise HTTPException(status_code=404, detail='job não encontrado')
    return json.loads(p.read_text(encoding='utf-8'))
def patch_bake_file():
    p=ROOT/'tsr'/'bake_texture.py'
    c=p.read_text(encoding='utf-8')
    c=c.replace('moderngl.create_context(standalone=True)', "moderngl.create_context(standalone=True, backend='egl')")
    c=c.replace('positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1])', 'positions = torch.tensor(positions_texture.reshape(-1, 4)[:, :-1], device=scene_code.device)')
    c=c.replace('rgb_f = queried_grid[\"color\"].numpy().reshape(-1, 3)', 'rgb_f = queried_grid[\"color\"].detach().cpu().numpy().reshape(-1, 3)')
    p.write_text(c, encoding='utf-8')
def convert_to_glb(job_dir, output_dir):
    target=job_dir/'model.glb'
    meshes=list(output_dir.rglob('*.obj'))+list(output_dir.rglob('*.glb'))
    if not meshes: raise RuntimeError('Nenhuma malha encontrada.')
    src=meshes[0]
    if src.suffix.lower()=='.glb': shutil.copy2(src,target)
    else:
        code=f"from pathlib import Path; import trimesh; s=trimesh.load(r'{src}', force='scene', process=False); s.export(r'{target}', file_type='glb')"
        r=subprocess.run([PYTHON,'-c',code],capture_output=True,text=True)
        if r.returncode!=0: raise RuntimeError(r.stderr)
    return target
def process_job(job_id,input_path,quality,remove_bg):
    d=JOBS_ROOT/job_id; out=d/'output'
    try:
        write_status(d,{'jobId':job_id,'status':'processing','message':'Gerando modelo 3D...','progress':20,'modelUrl':None})
        patch_bake_file()
        if out.exists(): shutil.rmtree(out)
        out.mkdir(parents=True,exist_ok=True)
        tex={'fast':'512','balanced':'1024','high':'2048'}.get(quality,'1024')
        cmd=[PYTHON,'run.py',input_path,'--output-dir',str(out),'--bake-texture','--texture-resolution',tex]
        if remove_bg: cmd.append('--remove-bg')
        env=os.environ.copy(); env['PYOPENGL_PLATFORM']='egl'
        r=subprocess.run(cmd,cwd=ROOT,capture_output=True,text=True,env=env)
        if r.returncode!=0: raise RuntimeError(r.stderr or 'Falha no TripoSR')
        write_status(d,{'jobId':job_id,'status':'processing','message':'Convertendo para GLB...','progress':80,'modelUrl':None})
        convert_to_glb(d,out)
        u=f'/models/{job_id}'
        write_status(d,{'jobId':job_id,'status':'done','message':'Modelo pronto.','progress':100,'modelUrl':u,'glbUrl':u,'url':u})
    except Exception as e:
        write_status(d,{'jobId':job_id,'status':'error','message':str(e),'progress':100,'modelUrl':None})
@app.get('/')
def root(): return {'ok':True,'service':'atlas-forge-engine-api','version':'0.4'}
@app.post('/image-to-3d')
async def image_to_3d(image:UploadFile=File(...),quality:str=Form('balanced'),removeBackground:str=Form('true'),prompt:str=Form('')):
    suf=Path(image.filename or 'upload.png').suffix.lower()
    if suf not in {'.png','.jpg','.jpeg','.webp'}: raise HTTPException(status_code=400,detail='Use PNG, JPG ou WEBP.')
    job=uuid.uuid4().hex[:12]; d=JOBS_ROOT/job; d.mkdir(parents=True,exist_ok=True); inp=d/f'input{suf}'; inp.write_bytes(await image.read())
    write_status(d,{'jobId':job,'status':'queued','message':'Job criado.','progress':5,'modelUrl':None})
    threading.Thread(target=process_job,args=(job,str(inp),quality,str(removeBackground).lower()!='false'),daemon=True).start()
    return {'jobId':job,'status':'queued','statusUrl':f'/image-to-3d-status/{job}','pollUrl':f'/image-to-3d-status/{job}'}
@app.get('/image-to-3d-status/{job_id}')
def status(job_id:str): return read_status(job_id)
@app.get('/models/{job_id}')
def model(job_id:str):
    p=JOBS_ROOT/job_id/'model.glb'
    if not p.exists(): raise HTTPException(status_code=404,detail='Modelo ainda não disponível.')
    return FileResponse(p,media_type='model/gltf-binary',filename=f'{job_id}.glb')
''', encoding='utf-8')
print('API criada:', APP_FILE)


## 4. Subir o servidor no Colab


In [ ]:
import subprocess, time
server = subprocess.Popen([PYTHON, '-m', 'uvicorn', 'atlas_forge_api:app', '--host', '0.0.0.0', '--port', '8000'], cwd='/content')
time.sleep(5)
print('Servidor PID:', server.pid)


## 5. Gerar URL pública


In [ ]:
from pyngrok import ngrok
ngrok.kill()
public_url = ngrok.connect(8000).public_url
print('COLE ESTA URL NO DEAD-ZONE:', public_url)
